In [1]:
import argparse
import logging
import os, sys
import torch
import numpy as np
import random
import json

import argparse
# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

sys.path.append(os.getcwd())

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo

def init_cfg():
    base_folder = '/Data_large/marine/PythonProjects/MMDET/MyConfigs'
    cfg = Config.fromfile(f'{base_folder}/Venus_b5/vfnet_r18.py')
    return cfg
    
def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)

In [2]:
def parse_args():
    # Argument Parser:
    parser = argparse.ArgumentParser(description="Trainer")
    
    def parse_band_list(band_str):
        """Parse a comma-separated string into a list of integers."""
        return [int(band) for band in band_str.split(',')]
    
    parser.add_argument('--band', type=parse_band_list, help='The band list num to train(list)', default='1,2,3')
    parser.add_argument('--seed', type=int, help='The seed', default=41)
    parser.add_argument('--batch_size', type=int, help='The batch size', default=3)
    parser.add_argument('--learning_rate', type=float, help='The learning rate', default=0.001)
    parser.add_argument('--resize', type=int, help='The resize dimension', default=2048)
    parser.add_argument('--random_crop', type=int, help='The resize dimension', default=None)
    
    # Simulate command line arguments
    sys.argv = ['ipykernel_launcher.py', 
                '--band', '3,4,7', 
                '--seed', '42', 
                '--batch_size', '2', 
                '--learning_rate', '0.001', 
                '--resize', '2048']
    
    # Use parse_known_args to handle unrecognized arguments
    args, unknown = parser.parse_known_args()
    return args

# Now call the function and print the arguments
args = parse_args()
print(args)

Namespace(band=[3, 4, 7], batch_size=2, learning_rate=0.001, random_crop=None, resize=2048, seed=42)


In [3]:
def main(args):
    setup_cache_size_limit_of_dynamo()
    
    cfg = init_cfg()
    MAX_EPOCHS = 20
    BAND_SEL = args.band # Selecting the Bands for Venus
    AMP = False
    # Normalization:
    MEANS=[158.69588,124.42161,109.27108,105.380424,88.40926,98.93067,88.819916,94.20678,103.540764,111.64337,122.92817,79.31501]
    STD=[34.95446,46.282494,56.252197,55.741932,64.54027,59.59095,69.65824,68.40028,77.930405,103.4634,105.30468,65.8369]
    
    MEAN_VALS = [MEANS[x-1] for x in BAND_SEL]
    STD_VALS = [STD[x-1] for x in BAND_SEL]
    # Resizing:
    IMG_SIZE = args.resize

    # Training:
    BS = args.batch_size
    LR = args.learning_rate

    # Annotations:
    ann_file = {'Train': f'/Data_large/marine/Datasets/VENuS/annotations/perfect/train__band_{BAND_SEL[0]}.json',
        'Val': f'/Data_large/marine/Datasets/VENuS/annotations/perfect/val__band_{BAND_SEL[0]}.json',
        'Test': f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{BAND_SEL[0]}.json',
        }

    ## Testing:
    # ann_file = '/Data_large/marine/Datasets/VENuS/annotations/perfect/test.json'
    data_root = '/Data_large/marine/Datasets/VENuS/ds_L0/'
    if len(BAND_SEL) == 1:
        data_prefix = f'perfect_b{BAND_SEL[0]}/'
    else:
        data_prefix = f'perfect/'

    # Deterministic Behaviour setting:
    SEED = args.seed
    set_seed(SEED)
    cfg.randomness = dict(
        seed = SEED, # 41 72 18
        diff_rank_seed=True,
        # deterministic=True
    )

    # Optimizers:
    optimizers =  {'SGD':{'type': 'OptimWrapper', 'optimizer': {'type': 'SGD', 'lr': LR, 'momentum': 0.9, 'weight_decay': 0.0001}},
                'Adam':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},
                'AdamW':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},}
    selOpt = 'SGD'

    # Savedir
    bandsNames = ''.join([f'_b{x}' for x in BAND_SEL])
    workdir = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/perfect{bandsNames}/{SEED}_BS_{BS}_LR_{LR}_ME_{MAX_EPOCHS}_OPT_{selOpt}'
    
        #### WORKDIR
    cfg.work_dir = workdir

    #### AMP
    # enable automatic-mixed-precision training
    if AMP is True:
        optim_wrapper = cfg.optim_wrapper.type
        if optim_wrapper == 'AmpOptimWrapper':
            print_log(
                'AMP training is already enabled in your config.',
                logger='current',
                level=logging.WARNING)
        else:
            assert optim_wrapper == 'OptimWrapper', (
                '`--amp` is only supported when the optimizer wrapper type is '
                f'`OptimWrapper` but got {optim_wrapper}.')
            cfg.optim_wrapper.type = 'AmpOptimWrapper'
            cfg.optim_wrapper.loss_scale = 'dynamic'

    # Dataloader:
    cfg.model.data_preprocessor = dict(
        mean=[float(x) for x in MEAN_VALS],
        pad_size_divisor=1,
        std=[float(x) for x in STD_VALS],
        type='MyPrePro')
    
    # Model Inputs:
    cfg.model.backbone.in_channels = len(BAND_SEL)
    
    # Annotation file:
    cfg.train_dataloader.dataset.ann_file = ann_file['Train']
    cfg.train_dataloader.dataset.data_prefix = {'img':data_prefix}
    cfg.train_dataloader.dataset.data_root = data_root
    
    cfg.val_dataloader.dataset.ann_file = ann_file['Train']
    cfg.val_dataloader.dataset.data_prefix = {'img':data_prefix}
    cfg.val_dataloader.dataset.data_root = data_root
    
    #       Evaluators:
    cfg.val_evaluator = dict(
        ann_file=ann_file['Val'],
        backend_args=None,
        format_only=False,
        metric='bbox',
        type='CocoMetric')
    #      Pipeline:
    #             Hook for custom loader:
    cfg.train_dataloader.dataset.pipeline[0] = {'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL}
    cfg.val_dataloader.dataset.pipeline[0] = {'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL}
    
    cfg.train_dataloader.dataset.pipeline[3] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}
    cfg.val_dataloader.dataset.pipeline[2] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}
    
    # Adding random crop to the pipeline. TODO: training with decreasing size
    if args.random_crop is not None:
        assert isinstance(args.random_crop, int), 'RandomCrop Error: single dimension must be specified. E.g. 224'
        # insert random crop: 
        rc = dict(type='RandomCrop', crop_size=(args.random_crop, args.random_crop))
        cfg.train_dataloader.dataset.pipeline.insert(3, rc)
        cfg.val_dataloader.dataset.pipeline.insert(2, rc)

    # Training params:
    cfg.train_dataloader.batch_size = BS
    cfg.train_cfg = {'type': 'EpochBasedTrainLoop', 'max_epochs': MAX_EPOCHS, 'val_interval': 1}

    # TODO: implement stages as in: https://github.com/open-mmlab/mmdetection/blob/cfd5d3a985b0249de009b67d04f37263e11cdf3d/configs/rtmdet/rtmdet_x_p6_4xb8-300e_coco.py#L78
    # lr_config = dict(policy='poly', power=0.9, min_lr=1e-4, by_epoch=False)

    cfg.optim_wrapper = optimizers[selOpt]

    cfg.param_scheduler = [{'type': 'LinearLR',
                            'start_factor': 0.001,
                            'by_epoch': True,
                            'begin': 0,
                            'end': 5},
                            {'type': 'MultiStepLR',
                            'begin': 0,
                            'end': MAX_EPOCHS//2,
                            'by_epoch': True,
                            'milestones': [MAX_EPOCHS//4, MAX_EPOCHS//3, MAX_EPOCHS//2],
                            'gamma': 0.75}, 
                            {# use cosine lr from 150 to 300 epoch
                            'type':'CosineAnnealingLR',
                            'eta_min':LR * 0.05,
                            'begin':MAX_EPOCHS // 2,
                            'end':MAX_EPOCHS,
                            'T_max':MAX_EPOCHS // 2,
                            'by_epoch':True,
                            'convert_to_iter_based':True,}
                            ]

    #### Test Config hooks:
    default_hooks = cfg.default_hooks
    if 'visualization' in default_hooks:
        visualization_hook = default_hooks['visualization']
        # Turn on visualization
        visualization_hook['draw'] = False

    cfg.test_dataloader = dict(
                batch_size=1,
                dataset=dict(
                    ann_file=ann_file['Test'],
                    data_root=data_root,
                    data_prefix=dict(img=data_prefix),
                    filter_cfg=dict(filter_empty_gt=True),
                    metainfo=dict(classes=('Vessel', ), palette=[
                        (
                            220,
                            20,
                            60,
                        ),
                    ]),
                    pipeline=[{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL},
                        dict(type='LoadAnnotations', with_bbox=True),
                        dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                        dict(
                            meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                    'height', 'width', 'instances', 'sample_idx', 
                                    'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                    'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                    'gt_bboxes_labels'),
                            type='PackDetInputs'),
                    ],
                    test_mode=True,
                    type='CocoDataset'),
                drop_last=False,
                num_workers=2,
                persistent_workers=True,
                sampler=dict(shuffle=False, type='DefaultSampler'))

    cfg.test_evaluator = dict(
                type='CocoMetric',
                metric='bbox',
                format_only=False,
                ann_file=ann_file['Test'],
                outfile_prefix=f'{workdir}/test_results')


    # build the runner from config
    if 'runner_type' not in cfg:
        # build the default runner
        runner = Runner.from_cfg(cfg)
    else:
        # build customized runner from the registry
        # if 'runner_type' is set in the cfg
        runner = RUNNERS.build(cfg)
    
    if RUN:
        ########## TRAINING:
        runner.train()
        ########## TESTING:
        runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{workdir}/test_result/test.pkl'))
        # start testing
        output_test_data =runner.test()

        # Specify the file name
        file_name = f'{workdir}/test_result/coco_metrics.json'# Specify the filepath
        # Write the dictionary to a JSON file
        with open(file_name, 'w') as json_file:
            json.dump(output_test_data, json_file, indent=4)

        print(f"Data has been saved to {file_name}")
    
    
RUN = True

main(args=parse_args())



08/02 10:18:03 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/02 10:18:18 - mmengine - INFO - Epoch(train)  [1][ 1/70]  lr: 1.0000e-06  eta: 3:02:59  time: 7.8480  data_time: 1.1144  memory: 5361  loss: 1.2131  loss_cls: 0.0068  loss_bbox: 0.5290  loss_bbox_rf: 0.6773
08/02 10:18:19 - mmengine - INFO - Epoch(train)  [1][ 2/70]  lr: 1.0000e-06  eta: 1:40:49  time: 4.3273  data_time: 0.5673  memory: 5437  loss: 1.5590  loss_cls: 0.0055  loss_bbox: 0.6816  loss_bbox_rf: 0.8719
08/02 10:18:19 - mmengine - INFO - Epoch(train)  [1][ 3/70]  lr: 1.0000e-06  eta: 1:13:00  time: 3.1359  data_time: 0.3849  memory: 5437  loss: 1.4211  loss_cls: 0.0061  loss_bbox: 0.6199  loss_bbox_rf: 0.7951
08/02 10:18:20 - mmengine - INFO - Epoch(train)  [1][ 4/70]  lr: 1.0000e-06  eta: 0:59:00  time: 2.5365  data_time: 0.2937  memory: 5437  loss: 1.6779  loss_cls: 0.0057  loss_bbox: 0.7211  loss_bbox_rf: 0.9512
08/02 10:18:21 - mmengine - INFO - Epoch(train)  [1][ 5/70]  lr: 1.0000e-06  eta: 0:51:22  time: 2.2095  data_time: 0.2401  memory: 5437  loss: 1.5784  loss_cls